# 作业1：自动拼写纠错

欢迎来到第2门课程的第一次作业。本次作业能够帮助你巩固 Python 编程与概率相关知识。通过完成本次作业，你将实现一套高效实用的自动拼写纠错系统。

## 大纲
- [0. 概述](#0)
    - [0.1 编辑距离](#0-1)
- [1. 数据预处理](#1)
    - [1.1 练习1](#ex-1)
    - [1.2 练习2](#ex-2)
    - [1.3 练习3](#ex-3)
- [2. 字符串操作](#2)
    - [2.1 练习4](#ex-4)
    - [2.2 练习5](#ex-5)
    - [2.3 练习6](#ex-6)
    - [2.4 练习7](#ex-7)
- [3. 编辑操作组合](#3)
    - [3.1 练习8](#ex-8)
    - [3.2 练习9](#ex-9)
    - [3.3 练习10](#ex-10)
- [4. 最小编辑距离](#4)
    - [4.1 练习11](#ex-11)
- [5. 回溯路径（选做）](#5)

<a name='0'></a>
## 0. 概述

大家每天在手机和电脑上都会使用自动纠错功能。在本次作业中，你将探究它背后真正的实现原理。当然，你即将实现的模型和手机内置的纠错系统并不完全一致，但效果依旧很不错。

完成本次作业后，你将学会：
- 根据语料统计单词出现次数
- 计算语料中单词的出现概率
- 进行字符串处理
- 筛选字符串
- 实现最小编辑距离算法，用于字符串比对并寻找最优编辑路径
- 理解动态规划的工作原理

同类系统应用十分广泛。
举个例子，如果你输入 **"I am lerningg"**，系统很大概率能判断出你实际想输入的是 **"learning"**，如图1所示。

<div style="width:image width px; font-size:100%; text-align:center;"><img src='auto-correct.png' alt="alternate text" width="width" height="height" style="width:300px;height:250px;" /> Figure 1 </div>

<a name='0-1'></a>
#### 0.1 编辑距离（Edit Distance）

在本作业中，你将实现能够纠正与目标词相差 1 个和 2 个编辑距离的单词的模型。
- 当我们需要对一个单词进行 n 次编辑才能将其变为另一个单词时，我们称这两个单词之间的编辑距离为 n。

一次编辑可以包含以下操作之一：

- 删除（移除一个字母）：'hat' => 'at, ha, ht'
- 交换（交换两个相邻字母）：'eta' => 'eat, tea, ...'
- 替换（将一个字母改为另一个字母）：'jat' => 'hat, rat, cat, mat, ...'
- 插入（添加一个字母）：'te' => 'the, ten, ate, ...'

你将使用上述四种方法来实现一个自动拼写校正器（Auto-correct）。
- 为此，你需要计算在给定输入条件下，某个特定单词是正确的概率。

你将要实现的这个自动拼写校正器最早由 [Peter Norvig](https://en.wikipedia.org/wiki/Peter_Norvig) 于 2007 年创建。
- 他的 [原始文章](https://norvig.com/spell-correct.html) 可能对本作业有参考价值。

我们拼写检查模型的目标是计算以下概率：

$$P(c|w) = \frac{P(w|c)\times P(c)}{P(w)} \tag{Eqn-1}$$

上述方程即为 [贝叶斯法则](https://en.wikipedia.org/wiki/Bayes%27_theorem)。
- 方程 1 表示：某个单词是正确的概率 $P(c|w)$，等于在它是正确的前提下出现特定单词 $w$ 的概率 $P(w|c)$，乘以该单词在一般情况下为正确的先验概率 $P(c)$，再除以该单词 $w$ 在一般情况下出现的概率 $P(w)$。
- 为了计算方程 1，你首先需要导入一个数据集，然后利用该数据集计算出所有所需的概率。

<a name='1'></a>
# Part 1: Data Preprocessing 

In [3]:
import re
from collections import Counter
import numpy as np
import pandas as pd

和其他所有机器学习任务一样，首要步骤就是处理数据集。
- 很多课程会直接提供预处理完毕的数据。
- 但在真实场景中搭建NLP系统时，需要自行加载并处理数据集。
- 接下来我们就实操体验真实场景下的数据预处理流程！

你的第一项任务：读取目录下名为 **'shakespeare.txt'** 的文件。
你可以通过菜单栏 `File ==> Open` 打开查看这份文件。

<a name='ex-1'></a>
### 练习1
实现函数 `process_data`，功能要求：
1）读取语料文本文件
2）将所有文本转为小写
3）返回单词组成的列表

#### 可选建议与提示
- 如果你希望获得更贴近真实工程场景的练习体验，暂时不要查看下方提示，尝试自行上网检索资料推导出解决方案。
- 需要少量指引时，可以鼠标点击绿色的「通用提示」区域展开阅读。
- 如果陷入瓶颈、运行结果达不到预期，可以点开绿色「详细提示」，获取完成该函数每一步操作对应的指引。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>General Hints</b></font>
</summary>
<p>
    
General Hints to get started
<ul>
    <li>Python <a href="https://docs.python.org/3/tutorial/inputoutput.html">input and output<a></li>
    <li>Python <a href="https://docs.python.org/3/library/re.html" >'re' documentation </a> </li>
</ul>
</p>


<details>    
<summary>
    <font size="3" color="darkgreen"><b>Detailed Hints</b></font>
</summary>
<p>     
Detailed hints if you're stuck
<ul>
    <li>Use 'with' syntax to read a file</li>
    <li>Decide whether to use 'read()' or 'readline().  What's the difference?</li>
    <li>Choose whether to use either str.lower() or str.lowercase().  What is the difference?</li>
    <li>Use re.findall(pattern, string)</li>
    <li>Look for the "Raw String Notation" section in the Python 're' documentation to understand the difference between r'\W', r'\W' and '\\W'. </li>
    <li>For the pattern, decide between using '\s', '\w', '\s+' or '\w+'.  What do you think are the differences?</li>
</ul>
</p>


In [4]:
# UNQ_C1 (唯一单元格标识，请勿修改)
# 计分函数：process_data
def process_data(file_name):
    """
    输入：
        file_name：当前目录下的文本文件名，读取该文件
    输出：
        words：列表，包含语料中全部转为小写后的单词
    """
    words = [] # 正确填充该变量并返回

    ### 代码开始 ###
    with open(file_name, 'r', encoding='utf-8') as f:
        text = f.read()
        # 使用正则表达式匹配单词，并将其转换为小写
        words = re.findall(r'\b\w+\b', text.lower())
    ### 代码结束 ###

    return words

Note, in the following cell, 'words' is converted to a python `set`. This eliminates any duplicate entries.

In [5]:
#DO NOT MODIFY THIS CELL
word_l = process_data('shakespeare.txt')
vocab = set(word_l)  # this will be your new vocabulary
print(f"The first ten words in the text are: \n{word_l[0:10]}")
print(f"There are {len(vocab)} unique words in the vocabulary.")

The first ten words in the text are: 
['o', 'for', 'a', 'muse', 'of', 'fire', 'that', 'would', 'ascend', 'the']
There are 6116 unique words in the vocabulary.


#### Expected Output
```Python
The first ten words in the text are: 
['o', 'for', 'a', 'muse', 'of', 'fire', 'that', 'would', 'ascend', 'the']
There are 6116 unique words in the vocabulary.
```

<a name='ex-2'></a>
### 练习2

实现一个 `get_count` 函数，该函数返回一个字典
- 字典的键是单词
- 每个单词对应的值是该单词在语料中出现的次数。

例如，给定如下句子：**"I am happy because I am learning"**，你的字典应当返回如下结果：
<table style="width:20%">

  <tr>
    <td> <b>键 </b>  </td>
    <td> <b>值 </b> </td> 


  </tr>
  <tr>
    <td> I  </td>
    <td> 2</td> 
 
  </tr>
   
  <tr>
    <td>am</td>
    <td>2</td> 
  </tr>

  <tr>
    <td>happy</td>
    <td>1</td> 
  </tr>
  
   <tr>
    <td>because</td>
    <td>1</td> 
  </tr>
  
   <tr>
    <td>learning</td>
    <td>1</td> 
  </tr>
</table>

**任务要求**：
实现 `get_count` 函数，返回一个字典，字典键为单词，值是单词在列表内出现的次数。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>Try implementing this using a for loop and a regular dictionary. This may be good practice for similar coding interview questions</li>
    <li>You can also use defaultdict instead of a regualr dictionary, along with the for loop</li>
    <li>Otherwise, to skip using a for loop, you can use Python's <a href="https://docs.python.org/3.7/library/collections.html#collections.Counter" > Counter class</a> </li>
</ul>
</p>

In [6]:
# UNQ_C2 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# UNIT TEST COMMENT: Candidate for Table Driven Tests
# GRADED FUNCTION: get_count
def get_count(word_l):
    '''
    Input:
        word_l: a set of words representing the corpus. 
    Output:
        word_count_dict: The wordcount dictionary where key is the word and value is its frequency.
    '''
    
    word_count_dict = {}  # fill this with word counts
    ### START CODE HERE 
    for word in word_l:
        word_count_dict[word] = word_count_dict.get(word, 0) + 1
    ### END CODE HERE ###
 
    return word_count_dict

In [7]:
#DO NOT MODIFY THIS CELL
word_count_dict = get_count(word_l)
print(f"There are {len(word_count_dict)} key values pairs")
print(f"The count for the word 'thee' is {word_count_dict.get('thee',0)}")

There are 6116 key values pairs
The count for the word 'thee' is 240



#### Expected Output
```Python
There are 6116 key values pairs
The count for the word 'thee' is 240
```

<a name='ex-3'></a>
### 练习3

给定单词计数字典，计算：从语料中随机抽取一个单词时，各个单词出现的概率。

$$P(w_i) = \frac{C(w_i)}{M} \tag{公式2}$$

其中：
$C(w_i)$ 代表单词 $w_i$ 在语料中出现的总次数；
$M$ 代表语料中所有单词的总数。

举个例子，在句子 **'I am happy because I am learning'** 中，单词 `am` 的概率为：

$$P(am) = \frac{C(w_i)}{M} = \frac {2}{7} \tag{公式3}$$

**任务要求：**
实现 `get_probs` 函数，用于求解单词在样本中出现的概率。函数返回字典：键为单词，值为该单词在语料中出现的概率。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
General advice
<ul>
    <li> Use dictionary.values() </li>
    <li> Use sum() </li>
    <li> The cardinality (number of words in the corpus should be equal to len(word_l).  You will calculate this same number, but using the word count dictionary.</li>
</ul>
    
If you're using a for loop:
<ul>
    <li> Use dictionary.keys() </li>
</ul>
    
If you're using a dictionary comprehension:
<ul>
    <li>Use dictionary.items() </li>
</ul>
</p>


In [8]:
# UNQ_C3 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: get_probs
def get_probs(word_count_dict):
    '''
    Input:
        word_count_dict: The wordcount dictionary where key is the word and value is its frequency.
    Output:
        probs: A dictionary where keys are the words and the values are the probability that a word will occur. 
    '''
    probs = {}  # return this variable correctly
    
    ### START CODE HERE ###
    M = sum(word_count_dict.values())  # Total number of words
    for word, count in word_count_dict.items():
        probs[word] = count / M  # Calculate probability
    ### END CODE HERE ###

    return probs

In [9]:
#DO NOT MODIFY THIS CELL
probs = get_probs(word_count_dict)
print(f"Length of probs is {len(probs)}")
print(f"P('thee') is {probs['thee']:.4f}")

Length of probs is 6116
P('thee') is 0.0045


#### Expected Output

```Python
Length of probs is 6116
P('thee') is 0.0045
```

<a name='2'></a>
# 第2部分：字符串操作

现在你已经计算出语料中所有单词的 $P(w_i)$。接下来你将编写若干字符串处理函数，用来对错误拼写的字符串进行编辑，生成正确拼写的候选单词。本节需要实现四个函数：

* `delete_letter`：输入一个单词，返回**删除任意一个字符后**得到的全部字符串。
* `switch_letter`：输入一个单词，返回**交换任意一对相邻字符位置**后得到的全部字符串。
* `replace_letter`：输入一个单词，返回**将任意一个字符替换为其他不同字母**后得到的全部字符串。
* `insert_letter`：输入一个单词，返回**在任意位置插入一个新字符**后得到的全部字符串。


#### 列表推导式

在 Python 中处理字符串与列表时常会用到一项语法特性：**列表推导式（list comprehensions）**。下方将要介绍的程序逻辑均采用列表推导式实现。当然，只要最终运行结果一致，你也可以选用其他实现方式。

后续内容会详细讲解如何使用列表推导式，并完成对应函数的编写。如果你已经熟练掌握 Python，可直接跳过相关提示，着手实现函数。

Python 的列表推导式能够把循环逻辑内嵌在列表定义当中，将多行代码压缩成一行。如果你尚不熟悉这种写法，相比普通 for 循环，第一眼看上去会觉得语序有些别扭。

<div style="width:image width px; font-size:100%; text-align:center;"><img src='GenericListComp3.PNG' alt="alternate text" width="width" height="height"  style="width:800px;height:400px;"/> Figure 2 </div>

The diagram above shows that the components of a list comprehension are the same components you would find in a typical for loop that appends to a list, but in a different order. With that in mind, we'll continue the specifics of this assignment. We will be very descriptive for the first function, `deletes()`, and less so in later functions as you become familiar with list comprehensions.

<a name='ex-4'></a>
### 练习4

**delete_letter() 实现要求：**
实现 `delete_letter()` 函数：输入一个单词，返回删除其中任意一个字符后得到的字符串列表。

举个例子，输入单词 **nice**，输出结果集合为：{'ice', 'nce', 'nic', 'nie'}。

**步骤1：** 创建 `splits` 列表。它包含将单词切分为左侧字符串 L 和右侧字符串 R 的所有切分方式。例如：
'nice' 的所有切分结果：`[('', 'nice'), ('n', 'ice'), ('ni', 'ce'), ('nic', 'e'), ('nice', '')]`

这种左右切分方式在接下来四个函数（删除、替换、交换、插入）中都会通用。

<div style="width:image width px; font-size:100%; text-align:center;"><img src='Splits1.PNG' alt="alternate text" width="width" height="height" style="width:650px;height:200px;" /> Figure 3 </div>

**步骤 2：** 这部分专门针对 `delete_letter`（删除字母）操作。在这里，我们要生成所有通过删除一个字符而得到的单词。  
这可以通过一行列表推导式来实现。你可以使用如下语法形式：  
`[f(a, b) for a, b in splits if condition]`  

以我们的示例 'nice' 为例，你会得到：  
['ice', 'nce', 'nie', 'nic']

<div style="width:image width px; font-size:100%; text-align:center;"><img src='ListComp2.PNG' alt="alternate text" width="width" height="height" style="width:550px;height:300px;" /> Figure 4 </div>

#### 辅助级别

请尝试按照以下辅助级别来完成此练习。  
- 我们希望这既能为你带来有意义的体验，又不会让你感到过于沮丧。
- 从第 1 级开始，然后根据需要进入第 2 级和第 3 级。

    - 第 1 级：尝试自己思考并独立实现。
    - 第 2 级：如果卡住了，可以点击“第 2 级提示”部分，获取一些入门提示。
    - 第 3 级：如果你希望获得更详细的指导，请点击“第 3 级提示”单元格，查看逐步操作说明。
    
- 如果仍然卡住，请参考上面“列表推导式”部分中的图片。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Level 2 Hints</b></font>
</summary>
<p>
<ul>
    <li><a href="" > Use array slicing like my_string[0:2] </a> </li>
    <li><a href="" > Use list comprehensions or for loops </a> </li>
</ul>
</p>


<details>    
<summary>
    <font size="3" color="darkgreen"><b>Level 3 Hints</b></font>
</summary>
<p>
<ul>
    <li>splits: Use array slicing, like my_str[0:2], to separate a string into two pieces.</li>
    <li>Do this in a loop or list comprehension, so that you have a list of tuples.
    <li> For example, "cake" can get split into "ca" and "ke". They're stored in a tuple ("ca","ke"), and the tuple is appended to a list.  We'll refer to these as L and R, so the tuple is (L,R)</li>
    <li>When choosing the range for your loop, if you input the word "cans" and generate the tuple  ('cans',''), make sure to include an if statement to check the length of that right-side string (R) in the tuple (L,R) </li>
    <li>deletes: Go through the list of tuples and combine the two strings together. You can use the + operator to combine two strings</li>
    <li>When combining the tuples, make sure that you leave out a middle character.</li>
    <li>Use array slicing to leave out the first character of the right substring.</li>
</ul>
</p>

In [20]:
# UNQ_C4 (唯一单元格标识符，请勿编辑)
# 单元测试注释：适用于表驱动测试的候选
# 计分函数：deletes
def delete_letter(word, verbose=False):
    '''
    输入参数：
        word: 你将为其生成词汇表中所有可能缺失一个字符的单词的字符串/单词
    输出：
        delete_l: 通过从 word 中删除 1 个字符而获得的所有可能字符串的列表
    '''
    
    delete_l = []
    split_l = []
    
    ### 在此处开始编写代码 ###
    split_l = [(word[:i],word[i:]) for i in range(0,len(word))]

    delete_l = [L + R[1:] for L, R in split_l if R]
    ### 在此处结束代码 ###


    if verbose: print(f"输入单词 {word}, \nsplit_l = {split_l}, \ndelete_l = {delete_l}")

    return delete_l

In [21]:
delete_word_l = delete_letter(word="cans",
                        verbose=True)

输入单词 cans, 
split_l = [('', 'cans'), ('c', 'ans'), ('ca', 'ns'), ('can', 's')], 
delete_l = ['ans', 'cns', 'cas', 'can']


#### 预期输出
```CPP
注意：你得到的 split_l 结果可能略有不同

输入单词 cans, 
split_l = [('', 'cans'), ('c', 'ans'), ('ca', 'ns'), ('can', 's')], 
delete_l = ['ans', 'cns', 'cas', 'can']

#### 备注 1
- 请注意，其中包含了额外的元组 `('cans', '')`。
- 只要你检查了元组 (L,R) 中右侧子字符串的长度，这不会有问题。
- 你能解释为什么这对于删除字符串列表（delete_l）会得到相同的结果吗？

```CPP
输入单词 cans, 
split_l = [('', 'cans'), ('c', 'ans'), ('ca', 'ns'), ('can', 's'), ('cans', '')], 
delete_l = ['ans', 'cns', 'cas', 'can']

#### 备注 2
如果你最终得到的单词与输入单词相同，就像这样：

```Python
输入单词 cans, 
split_l = [('', 'cans'), ('c', 'ans'), ('ca', 'ns'), ('can', 's'), ('cans', '')], 
delete_l = ['ans', 'cns', 'cas', 'can', 'cans']

In [22]:
# test # 2
print(f"Number of outputs of delete_letter('at') is {len(delete_letter('at'))}")

Number of outputs of delete_letter('at') is 2


#### Expected output

```CPP
Number of outputs of delete_letter('at') is 2
```

<a name='ex-5'></a>
### 练习 5

**switch_letter() 的说明**：现在实现一个函数，用于交换单词中的两个字母。它接受一个单词作为输入，并返回所有可能交换**相邻**两个字母后得到的单词列表。
- 例如，给定单词 'eta'，它返回 {'eat', 'tea'}，但不会返回 'ate'。

**步骤 1：** 与 delete_letter() 中的步骤 1 相同。  
**步骤 2：** 使用列表推导式或 for 循环，通过交换相邻字母来形成新字符串。其形式为：  
`[f(L,R) for L, R in splits if condition]`，其中 'condition' 将在给定的迭代中测试 R 的长度。详见下文。

<div style="width:image width px; font-size:100%; text-align:center;"><img src='Switches1.PNG' alt="alternate text" width="width" height="height" style="width:600px;height:200px;"/> Figure 5 </div>      

#### 难度级别

尝试按照以下难度级别来完成此练习。  
- 第 1 级：尝试自己思考并独立实现。  
- 第 2 级：如果卡住了，可以点击“第 2 级提示”部分，获取一些入门提示。  
- 第 3 级：如果你希望获得更详细的指导，请点击“第 3 级提示”单元格，查看逐步操作说明。

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Level 2 Hints</b></font>
</summary>
<p>
<ul>
    <li><a href="" > Use array slicing like my_string[0:2] </a> </li>
    <li><a href="" > Use list comprehensions or for loops </a> </li>
    <li>To do a switch, think of the whole word as divided into 4 distinct parts.  Write out 'cupcakes' on a piece of paper and see how you can split it into ('cupc', 'k', 'a', 'es')</li>
</ul>
</p>


<details>    
<summary>
    <font size="3" color="darkgreen"><b>Level 3 Hints</b></font>
</summary>
<p>
<ul>
    <li>splits: Use array slicing, like my_str[0:2], to separate a string into two pieces.</li>
    <li>Splitting is the same as for delete_letter</li>
    <li>To perform the switch, go through the list of tuples and combine four strings together. You can use the + operator to combine strings</li>
    <li>The four strings will be the left substring from the split tuple, followed by the first (index 1) character of the right substring, then the zero-th character (index 0) of the right substring, and then the remaining part of the right substring.</li>
    <li>Unlike delete_letter, you will want to check that your right substring is at least a minimum length.  To see why, review the previous hint bullet point (directly before this one).</li>
</ul>
</p>

In [ ]:
# UNQ_C5 (唯一单元格标识符，请勿编辑)
# 单元测试注释：适用于表驱动测试的候选
# 计分函数：switches
def switch_letter(word, verbose=False):
    '''
    输入参数：
        word: 输入字符串
    输出：
        switches: 所有可能交换一个相邻字符后得到的字符串列表
    ''' 
    
    switch_l = []
    split_l = []
    
    ### 在此处开始编写代码 ###
    ### 在此处结束代码 ###

    
    if verbose: print(f"输入单词 = {word} \nsplit_l = {split_l} \nswitch_l = {switch_l}") 

    return switch_l

In [ ]:
switch_word_l = switch_letter(word="eta",
                         verbose=True)

#### Expected output

```Python
Input word = eta 
split_l = [('', 'eta'), ('e', 'ta'), ('et', 'a')] 
switch_l = ['tea', 'eat']
```

#### Note 1

You may get this:
```Python
Input word = eta 
split_l = [('', 'eta'), ('e', 'ta'), ('et', 'a'), ('eta', '')] 
switch_l = ['tea', 'eat']
```
- Notice how it has the extra tuple `('eta', '')`.
- This is also correct.
- Can you think of why this is the case?

#### Note 2

If you get an error
```Python
IndexError: string index out of range
```
- Please see if you have checked the length of the strings when switching characters.

In [ ]:
# test # 2
print(f"Number of outputs of switch_letter('at') is {len(switch_letter('at'))}")

#### Expected output

```CPP
Number of outputs of switch_letter('at') is 1
```

<a name='ex-6'></a>
### Exercise 6
**Instructions for replace_letter()**: Now implement a function that takes in a word and returns a list of strings with one **replaced letter** from the original word. 

**Step 1:** is the same as in `delete_letter()`

**Step 2:** A list comprehension or for loop which form strings by replacing letters.  This can be of the form:  
`[f(a,b,c) for a, b in splits if condition for c in string]`   Note the use of the second for loop.  
It is expected in this routine that one or more of the replacements will include the original word. For example, replacing the first letter of 'ear' with 'e' will return 'ear'.

**Step 3:** Remove the original input letter from the output.

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>To remove a word from a list, first store its contents inside a set()</li>
    <li>Use set.discard('the_word') to remove a word in a set (if the word does not exist in the set, then it will not throw a KeyError.  Using set.remove('the_word') throws a KeyError if the word does not exist in the set. </li>
</ul>
</p>


In [ ]:
# UNQ_C6 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# UNIT TEST COMMENT: Candidate for Table Driven Tests
# GRADED FUNCTION: replaces
def replace_letter(word, verbose=False):
    '''
    Input:
        word: the input string/word 
    Output:
        replaces: a list of all possible strings where we replaced one letter from the original word. 
    ''' 
    
    letters = 'abcdefghijklmnopqrstuvwxyz'
    replace_l = []
    split_l = []
    
    ### START CODE HERE ###
    ### END CODE HERE ###

    
    # turn the set back into a list and sort it, for easier viewing
    replace_l = sorted(list(replace_set))
    
    if verbose: print(f"Input word = {word} \nsplit_l = {split_l} \nreplace_l {replace_l}")   
    
    return replace_l

In [ ]:
replace_l = replace_letter(word='can',
                              verbose=True)

#### Expected Output**: 
```Python
Input word = can 
split_l = [('', 'can'), ('c', 'an'), ('ca', 'n')] 
replace_l ['aan', 'ban', 'caa', 'cab', 'cac', 'cad', 'cae', 'caf', 'cag', 'cah', 'cai', 'caj', 'cak', 'cal', 'cam', 'cao', 'cap', 'caq', 'car', 'cas', 'cat', 'cau', 'cav', 'caw', 'cax', 'cay', 'caz', 'cbn', 'ccn', 'cdn', 'cen', 'cfn', 'cgn', 'chn', 'cin', 'cjn', 'ckn', 'cln', 'cmn', 'cnn', 'con', 'cpn', 'cqn', 'crn', 'csn', 'ctn', 'cun', 'cvn', 'cwn', 'cxn', 'cyn', 'czn', 'dan', 'ean', 'fan', 'gan', 'han', 'ian', 'jan', 'kan', 'lan', 'man', 'nan', 'oan', 'pan', 'qan', 'ran', 'san', 'tan', 'uan', 'van', 'wan', 'xan', 'yan', 'zan']
```
- Note how the input word 'can' should not be one of the output words.

#### Note 1
If you get something like this:

```Python
Input word = can 
split_l = [('', 'can'), ('c', 'an'), ('ca', 'n'), ('can', '')] 
replace_l ['aan', 'ban', 'caa', 'cab', 'cac', 'cad', 'cae', 'caf', 'cag', 'cah', 'cai', 'caj', 'cak', 'cal', 'cam', 'cao', 'cap', 'caq', 'car', 'cas', 'cat', 'cau', 'cav', 'caw', 'cax', 'cay', 'caz', 'cbn', 'ccn', 'cdn', 'cen', 'cfn', 'cgn', 'chn', 'cin', 'cjn', 'ckn', 'cln', 'cmn', 'cnn', 'con', 'cpn', 'cqn', 'crn', 'csn', 'ctn', 'cun', 'cvn', 'cwn', 'cxn', 'cyn', 'czn', 'dan', 'ean', 'fan', 'gan', 'han', 'ian', 'jan', 'kan', 'lan', 'man', 'nan', 'oan', 'pan', 'qan', 'ran', 'san', 'tan', 'uan', 'van', 'wan', 'xan', 'yan', 'zan']
```
- Notice how split_l has an extra tuple `('can', '')`, but the output is still the same, so this is okay.

#### Note 2
If you get something like this:
```Python
Input word = can 
split_l = [('', 'can'), ('c', 'an'), ('ca', 'n'), ('can', '')] 
replace_l ['aan', 'ban', 'caa', 'cab', 'cac', 'cad', 'cae', 'caf', 'cag', 'cah', 'cai', 'caj', 'cak', 'cal', 'cam', 'cana', 'canb', 'canc', 'cand', 'cane', 'canf', 'cang', 'canh', 'cani', 'canj', 'cank', 'canl', 'canm', 'cann', 'cano', 'canp', 'canq', 'canr', 'cans', 'cant', 'canu', 'canv', 'canw', 'canx', 'cany', 'canz', 'cao', 'cap', 'caq', 'car', 'cas', 'cat', 'cau', 'cav', 'caw', 'cax', 'cay', 'caz', 'cbn', 'ccn', 'cdn', 'cen', 'cfn', 'cgn', 'chn', 'cin', 'cjn', 'ckn', 'cln', 'cmn', 'cnn', 'con', 'cpn', 'cqn', 'crn', 'csn', 'ctn', 'cun', 'cvn', 'cwn', 'cxn', 'cyn', 'czn', 'dan', 'ean', 'fan', 'gan', 'han', 'ian', 'jan', 'kan', 'lan', 'man', 'nan', 'oan', 'pan', 'qan', 'ran', 'san', 'tan', 'uan', 'van', 'wan', 'xan', 'yan', 'zan']
```
- Notice how there are strings that are 1 letter longer than the original word, such as `cana`.
- Please check for the case when there is an empty string `''`, and if so, do not use that empty string when setting replace_l.

In [ ]:
# test # 2
print(f"Number of outputs of switch_letter('at') is {len(switch_letter('at'))}")

#### Expected output
```CPP
Number of outputs of switch_letter('at') is 1
```

<a name='ex-7'></a>
### Exercise 7

**Instructions for insert_letter()**: Now implement a function that takes in a word and returns a list with a letter inserted at every offset.

**Step 1:** is the same as in `delete_letter()`

**Step 2:** This can be a list comprehension of the form:  
`[f(a,b,c) for a, b in splits if condition for c in string]`   

In [ ]:
# UNQ_C7 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# UNIT TEST COMMENT: Candidate for Table Driven Tests
# GRADED FUNCTION: inserts
def insert_letter(word, verbose=False):
    '''
    Input:
        word: the input string/word 
    Output:
        inserts: a set of all possible strings with one new letter inserted at every offset
    ''' 
    letters = 'abcdefghijklmnopqrstuvwxyz'
    insert_l = []
    split_l = []
    
    ### START CODE HERE ###
    ### END CODE HERE ###


    if verbose: print(f"Input word {word} \nsplit_l = {split_l} \ninsert_l = {insert_l}")
    
    return insert_l

In [ ]:
insert_l = insert_letter('at', True)
print(f"Number of strings output by insert_letter('at') is {len(insert_l)}")

#### Expected output

```Python
Input word at 
split_l = [('', 'at'), ('a', 't'), ('at', '')] 
insert_l = ['aat', 'bat', 'cat', 'dat', 'eat', 'fat', 'gat', 'hat', 'iat', 'jat', 'kat', 'lat', 'mat', 'nat', 'oat', 'pat', 'qat', 'rat', 'sat', 'tat', 'uat', 'vat', 'wat', 'xat', 'yat', 'zat', 'aat', 'abt', 'act', 'adt', 'aet', 'aft', 'agt', 'aht', 'ait', 'ajt', 'akt', 'alt', 'amt', 'ant', 'aot', 'apt', 'aqt', 'art', 'ast', 'att', 'aut', 'avt', 'awt', 'axt', 'ayt', 'azt', 'ata', 'atb', 'atc', 'atd', 'ate', 'atf', 'atg', 'ath', 'ati', 'atj', 'atk', 'atl', 'atm', 'atn', 'ato', 'atp', 'atq', 'atr', 'ats', 'att', 'atu', 'atv', 'atw', 'atx', 'aty', 'atz']
Number of strings output by insert_letter('at') is 78
```

#### Note 1

If you get a split_l like this:
```Python
Input word at 
split_l = [('', 'at'), ('a', 't')] 
insert_l = ['aat', 'bat', 'cat', 'dat', 'eat', 'fat', 'gat', 'hat', 'iat', 'jat', 'kat', 'lat', 'mat', 'nat', 'oat', 'pat', 'qat', 'rat', 'sat', 'tat', 'uat', 'vat', 'wat', 'xat', 'yat', 'zat', 'aat', 'abt', 'act', 'adt', 'aet', 'aft', 'agt', 'aht', 'ait', 'ajt', 'akt', 'alt', 'amt', 'ant', 'aot', 'apt', 'aqt', 'art', 'ast', 'att', 'aut', 'avt', 'awt', 'axt', 'ayt', 'azt']
Number of strings output by insert_letter('at') is 52
```
- Notice that split_l is missing the extra tuple ('at', '').  For insertion, we actually **WANT** this tuple.
- The function is not creating all the desired output strings.
- Check the range that you use for the for loop.

#### Note 2
If you see this:
```Python
Input word at 
split_l = [('', 'at'), ('a', 't'), ('at', '')] 
insert_l = ['aat', 'bat', 'cat', 'dat', 'eat', 'fat', 'gat', 'hat', 'iat', 'jat', 'kat', 'lat', 'mat', 'nat', 'oat', 'pat', 'qat', 'rat', 'sat', 'tat', 'uat', 'vat', 'wat', 'xat', 'yat', 'zat', 'aat', 'abt', 'act', 'adt', 'aet', 'aft', 'agt', 'aht', 'ait', 'ajt', 'akt', 'alt', 'amt', 'ant', 'aot', 'apt', 'aqt', 'art', 'ast', 'att', 'aut', 'avt', 'awt', 'axt', 'ayt', 'azt']
Number of strings output by insert_letter('at') is 52
```

- Even though you may have fixed the split_l so that it contains the tuple `('at', '')`, notice that you're still missing some output strings.
    - Notice that it's missing strings such as 'ata', 'atb', 'atc' all the way to 'atz'.
- To fix this, make sure that when you set insert_l, you allow the use of the empty string `''`.

In [ ]:
# test # 2
print(f"Number of outputs of insert_letter('at') is {len(insert_letter('at'))}")

#### Expected output

```CPP
Number of outputs of insert_letter('at') is 78
```

<a name='3'></a>

# Part 3: Combining the edits

Now that you have implemented the string manipulations, you will create two functions that, given a string, will return all the possible single and double edits on that string. These will be `edit_one_letter()` and `edit_two_letters()`.

<a name='3-1'></a>
## 3.1 Edit one letter

<a name='ex-8'></a>
### Exercise 8

**Instructions**: Implement the `edit_one_letter` function to get all the possible edits that are one edit away from a word. The edits  consist of the replace, insert, delete, and optionally the switch operation. You should use the previous functions you have already implemented to complete this function. The 'switch' function  is a less common edit function, so its use will be selected by an "allow_switches" input argument.

Note that those functions return *lists* while this function should return a *python set*. Utilizing a set eliminates any duplicate entries.

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li> Each of the functions returns a list.  You can combine lists using the `+` operator. </li>
    <li> To get unique strings (avoid duplicates), you can use the set() function. </li>
</ul>
</p>


In [ ]:
# UNQ_C8 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# UNIT TEST COMMENT: Candidate for Table Driven Tests
# GRADED FUNCTION: edit_one_letter
def edit_one_letter(word, allow_switches = True):
    """
    Input:
        word: the string/word for which we will generate all possible wordsthat are one edit away.
    Output:
        edit_one_set: a set of words with one possible edit. Please return a set. and not a list.
    """
    
    edit_one_set = set()
    
    ### START CODE HERE ###
    ### END CODE HERE ###


    return edit_one_set

In [ ]:
tmp_word = "at"
tmp_edit_one_set = edit_one_letter(tmp_word)
# turn this into a list to sort it, in order to view it
tmp_edit_one_l = sorted(list(tmp_edit_one_set))

print(f"input word {tmp_word} \nedit_one_l \n{tmp_edit_one_l}\n")
print(f"The type of the returned object should be a set {type(tmp_edit_one_set)}")
print(f"Number of outputs from edit_one_letter('at') is {len(edit_one_letter('at'))}")

#### Expected Output
```CPP
input word at 
edit_one_l 
['a', 'aa', 'aat', 'ab', 'abt', 'ac', 'act', 'ad', 'adt', 'ae', 'aet', 'af', 'aft', 'ag', 'agt', 'ah', 'aht', 'ai', 'ait', 'aj', 'ajt', 'ak', 'akt', 'al', 'alt', 'am', 'amt', 'an', 'ant', 'ao', 'aot', 'ap', 'apt', 'aq', 'aqt', 'ar', 'art', 'as', 'ast', 'ata', 'atb', 'atc', 'atd', 'ate', 'atf', 'atg', 'ath', 'ati', 'atj', 'atk', 'atl', 'atm', 'atn', 'ato', 'atp', 'atq', 'atr', 'ats', 'att', 'atu', 'atv', 'atw', 'atx', 'aty', 'atz', 'au', 'aut', 'av', 'avt', 'aw', 'awt', 'ax', 'axt', 'ay', 'ayt', 'az', 'azt', 'bat', 'bt', 'cat', 'ct', 'dat', 'dt', 'eat', 'et', 'fat', 'ft', 'gat', 'gt', 'hat', 'ht', 'iat', 'it', 'jat', 'jt', 'kat', 'kt', 'lat', 'lt', 'mat', 'mt', 'nat', 'nt', 'oat', 'ot', 'pat', 'pt', 'qat', 'qt', 'rat', 'rt', 'sat', 'st', 't', 'ta', 'tat', 'tt', 'uat', 'ut', 'vat', 'vt', 'wat', 'wt', 'xat', 'xt', 'yat', 'yt', 'zat', 'zt']

The type of the returned object should be a set <class 'set'>
Number of outputs from edit_one_letter('at') is 129
```

<a name='3-2'></a>
## Part 3.2 Edit two letters

<a name='ex-9'></a>
### Exercise 9

Now you can generalize this to implement to get two edits on a word. To do so, you would have to get all the possible edits on a single word and then for each modified word, you would have to modify it again. 

**Instructions**: Implement the `edit_two_letters` function that returns a set of words that are two edits away. Note that creating additional edits based on the `edit_one_letter` function may 'restore' some one_edits to zero or one edits. That is allowed here. This accounted for in get_corrections.

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>You will likely want to take the union of two sets.</li>
    <li>You can either use set.union() or use the '|' (or operator) to union two sets</li>
    <li>See the documentation <a href="https://docs.python.org/2/library/sets.html" > Python sets </a> for examples of using operators or functions of the Python set.</li>
</ul>
</p>


In [ ]:
# UNQ_C9 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# UNIT TEST COMMENT: Candidate for Table Driven Tests
# GRADED FUNCTION: edit_two_letters
def edit_two_letters(word, allow_switches = True):
    '''
    Input:
        word: the input string/word 
    Output:
        edit_two_set: a set of strings with all possible two edits
    '''
    
    edit_two_set = set()
    
    ### START CODE HERE ###
    ### END CODE HERE ###

    
    return edit_two_set

In [ ]:
tmp_edit_two_set = edit_two_letters("a")
tmp_edit_two_l = sorted(list(tmp_edit_two_set))
print(f"Number of strings with edit distance of two: {len(tmp_edit_two_l)}")
print(f"First 10 strings {tmp_edit_two_l[:10]}")
print(f"Last 10 strings {tmp_edit_two_l[-10:]}")
print(f"The data type of the returned object should be a set {type(tmp_edit_two_set)}")
print(f"Number of strings that are 2 edit distances from 'at' is {len(edit_two_letters('at'))}")

#### Expected Output

```CPP
Number of strings with edit distance of two: 2654
First 10 strings ['', 'a', 'aa', 'aaa', 'aab', 'aac', 'aad', 'aae', 'aaf', 'aag']
Last 10 strings ['zv', 'zva', 'zw', 'zwa', 'zx', 'zxa', 'zy', 'zya', 'zz', 'zza']
The data type of the returned object should be a set <class 'set'>
Number of strings that are 2 edit distances from 'at' is 7154
```

<a name='3-3'></a>
## Part 3-3: suggest spelling suggestions

Now you will use your `edit_two_letters` function to get a set of all the possible 2 edits on your word. You will then use those strings to get the most probable word you meant to type aka your typing suggestion.

<a name='ex-10'></a>
### Exercise 10
**Instructions**: Implement `get_corrections`, which returns a list of zero to n possible suggestion tuples of the form (word, probability_of_word). 

**Step 1:** Generate suggestions for a supplied word: You'll use the edit functions you have developed. The 'suggestion algorithm' should follow this logic: 
* If the word is in the vocabulary, suggest the word. 
* Otherwise, if there are suggestions from `edit_one_letter` that are in the vocabulary, use those. 
* Otherwise, if there are suggestions from `edit_two_letters` that are in the vocabulary, use those. 
* Otherwise, suggest the input word.*  
* The idea is that words generated from fewer edits are more likely than words with more edits.


Note: 
- Edits of one or two letters may 'restore' strings to either zero or one edit. This algorithm accounts for this by preferentially selecting lower distance edits first.

#### Short circuit
In Python, logical operations such as `and` and `or` have two useful properties. They can operate on lists and they have ['short-circuit' behavior](https://docs.python.org/3/library/stdtypes.html). Try these:

In [ ]:
# example of logical operation on lists or sets
print( [] and ["a","b"] )
print( [] or ["a","b"] )
#example of Short circuit behavior
val1 =  ["Most","Likely"] or ["Less","so"] or ["least","of","all"]  # selects first, does not evalute remainder
print(val1)
val2 =  [] or [] or ["least","of","all"] # continues evaluation until there is a non-empty list
print(val2)

The logical `or` could be used to implement the suggestion algorithm very compactly. Alternately, if/then constructs could be used.
 
**Step 2**: Create a 'best_words' dictionary where the 'key' is a suggestion and the 'value' is the probability of that word in your vocabulary. If the word is not in the vocabulary, assign it a probability of 0.

**Step 3**: Select the n best suggestions. There may be fewer than n.

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>edit_one_letter and edit_two_letters return *python sets*. </li>
    <li> Sets have a handy <a href="https://docs.python.org/2/library/sets.html" > set.intersection </a> feature</li>
    <li>To find the keys that have the highest values in a dictionary, you can use the Counter dictionary to create a Counter object from a regular dictionary.  Then you can use Counter.most_common(n) to get the n most common keys.
    </li>
    <li>To find the intersection of two sets, you can use set.intersection or the & operator.</li>
    <li>If you are not as familiar with short circuit syntax (as shown above), feel free to use if else statements instead.</li>
    <li>To use an if statement to check of a set is empty, use 'if not x:' syntax </li>
</ul>
</p>


In [ ]:
# UNQ_C10 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# UNIT TEST COMMENT: Candidate for Table Driven Tests
# GRADED FUNCTION: get_corrections
def get_corrections(word, probs, vocab, n=2, verbose = False):
    '''
    Input: 
        word: a user entered string to check for suggestions
        probs: a dictionary that maps each word to its probability in the corpus
        vocab: a set containing all the vocabulary
        n: number of possible word corrections you want returned in the dictionary
    Output: 
        n_best: a list of tuples with the most probable n corrected words and their probabilities.
    '''
    
    suggestions = []
    n_best = []
    
    ### START CODE HERE ###
    ### END CODE HERE ###

    
    if verbose: print("entered word = ", word, "\nsuggestions = ", suggestions)

    return n_best

In [ ]:
# Test your implementation - feel free to try other words in my word
my_word = 'dys' 
tmp_corrections = get_corrections(my_word, probs, vocab, 2, verbose=True) # keep verbose=True
for i, word_prob in enumerate(tmp_corrections):
    print(f"word {i}: {word_prob[0]}, probability {word_prob[1]:.6f}")

print(f"data type of corrections {type(tmp_corrections)}")

#### Expected Output
- Note: This expected output is for `my_word = 'dys'`. Also, keep `verbose=True`
```CPP
entered word =  dys 
suggestions =  {'days', 'dye'}
word 0: days, probability 0.000410
word 1: dye, probability 0.000019
data type of corrections <class 'list'>
```

<a name='4'></a>
# Part 4: Minimum Edit distance

Now that you have implemented your auto-correct, how do you evaluate the similarity between two strings? For example: 'waht' and 'what'

Also how do you efficiently find the shortest path to go from the word, 'waht' to the word 'what'?

You will implement a dynamic programming system that will tell you the minimum number of edits required to convert a string into another string.

<a name='4-1'></a>
### Part 4.1 Dynamic Programming

Dynamic Programming breaks a problem down into subproblems which can be combined to form the final solution. Here, given a string source[0..i] and a string target[0..j], we will compute all the combinations of substrings[i, j] and calculate their edit distance. To do this efficiently, we will use a table to maintain the previously computed substrings and use those to calculate larger substrings.

You have to create a matrix and update each element in the matrix as follows:  

$$\text{Initialization}$$

\begin{align}
D[0,0] &= 0 \\
D[i,0] &= D[i-1,0] + del\_cost(source[i]) \tag{4}\\
D[0,j] &= D[0,j-1] + ins\_cost(target[j]) \\
\end{align}


$$\text{Per Cell Operations}$$
\begin{align}
 \\
D[i,j] =min
\begin{cases}
D[i-1,j] + del\_cost\\
D[i,j-1] + ins\_cost\\
D[i-1,j-1] + \left\{\begin{matrix}
rep\_cost; & if src[i]\neq tar[j]\\
0 ; & if src[i]=tar[j]
\end{matrix}\right.
\end{cases}
\tag{5}
\end{align}

So converting the source word **play** to the target word **stay**, using an input cost of one, a delete cost of 1, and replace cost of 2 would give you the following table:
<table style="width:20%">

  <tr>
    <td> <b> </b>  </td>
    <td> <b># </b>  </td>
    <td> <b>s </b>  </td>
    <td> <b>t </b> </td> 
    <td> <b>a </b> </td> 
    <td> <b>y </b> </td> 
  </tr>
   <tr>
    <td> <b>  #  </b></td>
    <td> 0</td> 
    <td> 1</td> 
    <td> 2</td> 
    <td> 3</td> 
    <td> 4</td> 
 
  </tr>
  <tr>
    <td> <b>  p  </b></td>
    <td> 1</td> 
 <td> 2</td> 
    <td> 3</td> 
    <td> 4</td> 
   <td> 5</td>
  </tr>
   
  <tr>
    <td> <b> l </b></td>
    <td>2</td> 
    <td>3</td> 
    <td>4</td> 
    <td>5</td> 
    <td>6</td>
  </tr>

  <tr>
    <td> <b> a </b></td>
    <td>3</td> 
     <td>4</td> 
     <td>5</td> 
     <td>4</td>
     <td>5</td> 
  </tr>
  
   <tr>
    <td> <b> y </b></td>
    <td>4</td> 
      <td>5</td> 
     <td>6</td> 
     <td>5</td>
     <td>4</td> 
  </tr>
  

</table>



The operations used in this algorithm are 'insert', 'delete', and 'replace'. These correspond to the functions that you defined earlier: insert_letter(), delete_letter() and replace_letter(). switch_letter() is not used here.

The diagram below describes how to initialize the table. Each entry in D[i,j] represents the minimum cost of converting string source[0:i] to string target[0:j]. The first column is initialized to represent the cumulative cost of deleting the source characters to convert string "EER" to "". The first row is initialized to represent the cumulative cost of inserting the target characters to convert from "" to "NEAR".

<div style="width:image width px; font-size:100%; text-align:center;"><img src='EditDistInit4.PNG' alt="alternate text" width="width" height="height" style="width:1000px;height:400px;"/> Figure 6 Initializing Distance Matrix</div>     

Filling in the remainder of the table utilizes the 'Per Cell Operations' in the equation (5) above. Note, the diagram below includes in the table some of the 3 sub-calculations shown in light grey. Only 'min' of those operations is stored in the table in the `min_edit_distance()` function.

<div style="width:image width px; font-size:100%; text-align:center;"><img src='EditDistFill2.PNG' alt="alternate text" width="width" height="height" style="width:800px;height:400px;"/> Figure 7 Filling Distance Matrix</div>     

Note that the formula for $D[i,j]$ shown in the image is equivalent to:

\begin{align}
 \\
D[i,j] =min
\begin{cases}
D[i-1,j] + del\_cost\\
D[i,j-1] + ins\_cost\\
D[i-1,j-1] + \left\{\begin{matrix}
rep\_cost; & if src[i]\neq tar[j]\\
0 ; & if src[i]=tar[j]
\end{matrix}\right.
\end{cases}
\tag{5}
\end{align}

The variable `sub_cost` (for substitution cost) is the same as `rep_cost`; replacement cost.  We will stick with the term "replace" whenever possible.

Below are some examples of cells where replacement is used. This also shows the minimum path from the lower right final position where "EER" has been replaced by "NEAR" back to the start. This provides a starting point for the optional 'backtrace' algorithm below.

<div style="width:image width px; font-size:100%; text-align:center;"><img src='EditDistExample1.PNG' alt="alternate text" width="width" height="height" style="width:1200px;height:400px;"/> Figure 8 Examples Distance Matrix</div>    

<a name='ex-11'></a>
### Exercise 11

Again, the word "substitution" appears in the figure, but think of this as "replacement".

**Instructions**: Implement the function below to get the minimum amount of edits required given a source string and a target string. 

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>The range(start, stop, step) function excludes 'stop' from its output</li>
    <li><a href="" > words </a> </li>
</ul>
</p>


In [ ]:
# UNQ_C11 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: min_edit_distance
def min_edit_distance(source, target, ins_cost = 1, del_cost = 1, rep_cost = 2):
    '''
    Input: 
        source: a string corresponding to the string you are starting with
        target: a string corresponding to the string you want to end with
        ins_cost: an integer setting the insert cost
        del_cost: an integer setting the delete cost
        rep_cost: an integer setting the replace cost
    Output:
        D: a matrix of len(source)+1 by len(target)+1 containing minimum edit distances
        med: the minimum edit distance (med) required to convert the source string to the target
    '''
    # use deletion and insert cost as  1
    m = len(source) 
    n = len(target) 
    #initialize cost matrix with zeros and dimensions (m+1,n+1) 
    D = np.zeros((m+1, n+1), dtype=int) 
    
    ### START CODE HERE (Replace instances of 'None' with your code) ###
    ### END CODE HERE ###

    return D, med

In [ ]:
#DO NOT MODIFY THIS CELL
# testing your implementation 
source =  'play'
target = 'stay'
matrix, min_edits = min_edit_distance(source, target)
print("minimum edits: ",min_edits, "\n")
idx = list('#' + source)
cols = list('#' + target)
df = pd.DataFrame(matrix, index=idx, columns= cols)
print(df)

**Expected Results:**  

```CPP
minimum edits:  4
    
   #  s  t  a  y
#  0  1  2  3  4
p  1  2  3  4  5
l  2  3  4  5  6
a  3  4  5  4  5
y  4  5  6  5  4
```

In [ ]:
#DO NOT MODIFY THIS CELL
# testing your implementation 
source =  'eer'
target = 'near'
matrix, min_edits = min_edit_distance(source, target)
print("minimum edits: ",min_edits, "\n")
idx = list(source)
idx.insert(0, '#')
cols = list(target)
cols.insert(0, '#')
df = pd.DataFrame(matrix, index=idx, columns= cols)
print(df)

**Expected Results**  
```CPP
minimum edits:  3 

   #  n  e  a  r
#  0  1  2  3  4
e  1  2  1  2  3
e  2  3  2  3  4
r  3  4  3  4  3
```

We can now test several of our routines at once:

In [ ]:
source = "eer"
targets = edit_one_letter(source,allow_switches = False)  #disable switches since min_edit_distance does not include them
for t in targets:
    _, min_edits = min_edit_distance(source, t,1,1,1)  # set ins, del, sub costs all to one
    if min_edits != 1: print(source, t, min_edits)

**Expected Results**  
```CPP
(empty)
```

The 'replace()' routine utilizes all letters a-z one of which returns the original word.

In [ ]:
source = "eer"
targets = edit_two_letters(source,allow_switches = False) #disable switches since min_edit_distance does not include them
for t in targets:
    _, min_edits = min_edit_distance(source, t,1,1,1)  # set ins, del, sub costs all to one
    if min_edits != 2 and min_edits != 1: print(source, t, min_edits)

**Expected Results**  
```CPP
eer eer 0
```

We have to allow single edits here because some two_edits will restore a single edit.

# Submission
Make sure you submit your assignment before you modify anything below


<a name='5'></a>

# Part 5: Optional - Backtrace


Once you have computed your matrix using minimum edit distance, how would find the shortest path from the top left corner to the bottom right corner? 

Note that you could use backtrace algorithm.  Try to find the shortest path given the matrix that your `min_edit_distance` function returned.

You can use these [lecture slides on minimum edit distance](https://web.stanford.edu/class/cs124/lec/med.pdf) by Dan Jurafsky to learn about the algorithm for backtrace.

In [ ]:
# Experiment with back trace - insert your code here



#### References
- Dan Jurafsky - Speech and Language Processing - Textbook
- This auto-correct explanation was first done by Peter Norvig in 2007 